# 1. Run PyNNLF ASHD 148-Household Experiments

This notebook runs the ASHD 148-household weather dataset through the same 12-model set used in the current Excel result, across three forecast horizons:

- `fh1`: 30 minutes
- `fh8`: 1 day
- `fh10`: 1 week

The notebook is resumable. It skips experiment combinations that already exist in `publication/journal_article_1/experiment_result` and enables PyNNLF figure generation via `plot_enabled=True`.


## 1. Setup

Find the publication workspace, add the local PyNNLF source path, and load the batch specification.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml


def find_publication_project(start: Path) -> Path:
    """Find publication/journal_article_1 from the current working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "specs" / "ashd_148hh_batch.yaml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    """Find the PyNNLF repository root."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project(Path.cwd().resolve())
REPO_ROOT = find_repo_root(PROJECT_DIR)
sys.path.insert(0, str(REPO_ROOT / "src"))

import pynnlf  # noqa: E402

BATCH_PATH = PROJECT_DIR / "specs" / "ashd_148hh_batch.yaml"
RESULTS_ROOT = PROJECT_DIR / "experiment_result"
TEMP_SPEC_PATH = PROJECT_DIR / "specs" / "_tmp_ashd_148hh_single.yaml"

batch = yaml.safe_load(BATCH_PATH.read_text(encoding="utf-8"))
config = yaml.safe_load((PROJECT_DIR / "specs" / "pynnlf_config.yaml").read_text(encoding="utf-8"))
forecast_horizon_minutes = {key: int(value) for key, value in config["forecast_horizons"].items()}

print(f"Publication project: {PROJECT_DIR}")
print(f"PyNNLF repo root: {REPO_ROOT}")
print(batch)


## 2. Validate input dataset

Confirm that `ds20_ashd_148hh_with_weather.csv` is ready for PyNNLF and includes the three weather variables.

In [ ]:
DATASET_FILE = PROJECT_DIR / "data" / "ds20_ashd_148hh_with_weather.csv"
EXPECTED_COLUMNS = [
    "datetime",
    "netload_kW",
    "air_temperature_in_degrees_c",
    "relative_humidity_in_percentage",
    "wind_speed_in_km_h",
]

if not DATASET_FILE.exists():
    raise FileNotFoundError(DATASET_FILE)

df = pd.read_csv(DATASET_FILE, parse_dates=["datetime"])
if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected columns in {DATASET_FILE.name}: {list(df.columns)}")
if len(df) != 52_608:
    raise ValueError(f"Expected 52,608 rows, found {len(df):,}")
if df["datetime"].duplicated().any():
    raise ValueError("Dataset contains duplicate timestamps")
if df.isna().any().any():
    raise ValueError("Dataset contains missing values")
freq = df["datetime"].diff().dropna().unique()
if len(freq) != 1 or freq[0] != pd.Timedelta(minutes=30):
    raise ValueError("Dataset is not regular 30-minute data")

print(f"{DATASET_FILE.name}: OK")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Rows: {len(df):,}")
display(df.head())


## 3. Run missing experiments

The completed-key check includes dataset, forecast horizon, model, and hyperparameter. This matters because the same model set is run at three horizons.

In [ ]:
def completed_keys(results_root: Path) -> set[tuple[str, int, str, str]]:
    """Return completed `(dataset_id, horizon_minutes, model_id, hp)` keys from existing result folders."""
    keys = set()
    for result_file in sorted(results_root.glob("E*/E*_a1_experiment_result.csv")):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
        except Exception:
            continue
        dataset_id = str(row.get("dataset_no", ""))
        model_id = str(row.get("model_no", ""))
        hp = str(row.get("hyperparameter_no", ""))
        try:
            horizon = int(row.get("forecast_horizon_min"))
        except Exception:
            continue
        if dataset_id and model_id and hp:
            keys.add((dataset_id, horizon, model_id, hp))
    return keys


done = completed_keys(RESULTS_ROOT)
total = len(batch["datasets"]) * len(batch["forecast_horizons"]) * len(batch["model_and_hp"])
run_index = 0

for dataset_id in batch["datasets"]:
    for forecast_horizon_id in batch["forecast_horizons"]:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp in batch["model_and_hp"]:
            run_index += 1
            key = (str(dataset_id), horizon_minutes, str(model_id), str(hp))
            if key in done:
                print(f"[skip {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
                continue

            print(f"[run {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
            temp_spec = {
                "datasets": [dataset_id],
                "forecast_horizons": [forecast_horizon_id],
                "model_and_hp": [[model_id, hp]],
            }
            TEMP_SPEC_PATH.write_text(yaml.safe_dump(temp_spec, sort_keys=False), encoding="utf-8")
            pynnlf.run_experiment_batch(TEMP_SPEC_PATH, plot_enabled=True)
            done.add(key)

if TEMP_SPEC_PATH.exists():
    TEMP_SPEC_PATH.unlink()

pynnlf.recap_experiments(RESULTS_ROOT)
print(f"Recap written to {RESULTS_ROOT / 'a1_experiment_result.csv'}")


## 4. Inspect ASHD recap rows

After the run, the recap should contain 36 `ds20` rows: 12 models times 3 forecast horizons.

In [ ]:
recap_path = RESULTS_ROOT / "a1_experiment_result.csv"
recap = pd.read_csv(recap_path)
ashd_recap = recap.loc[recap["dataset_no"].eq("ds20")].copy()
ashd_recap = ashd_recap.sort_values(["forecast_horizon_min", "model_name"])
print(f"ASHD ds20 recap rows: {len(ashd_recap):,}")
display(ashd_recap[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])
